In [ ]:
import os

In [4]:
import weaviate

weaviate_client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.getenv("WEAVIATE_URL"),
    auth_credentials=weaviate.auth.AuthApiKey(os.getenv("WEAVIATE_API_KEY")),
)

weaviate_client.is_ready()

True

In [9]:
from pydantic import BaseModel
from weaviate.classes.query import Filter

class DocWithDatasetID(BaseModel):
    doc: str
    dataset_id: str

def hybrid_search(
    weaviate_collection: weaviate.collections.Collection,
    query: str,
    num_results: int = 1,
) -> list[DocWithDatasetID]:
    response = weaviate_collection.query.hybrid(
        query=query,
        limit=num_results,
    )
    returned_objs: list[DocWithDatasetID] = []
    for o in response.objects:
        returned_objs.append(
            DocWithDatasetID(
                doc=o.properties["content"],
                dataset_id=o.properties["dataset_id"],
            )
        )
    return returned_objs

def fetch_gold_doc(
    weaviate_collection: weaviate.collections.Collection,
    gold_id: str,
) -> DocWithDatasetID:
    response = weaviate_collection.query.fetch_objects(
        filters=Filter.by_property("dataset_id").like(gold_id),
    )
    return DocWithDatasetID(
        doc=response.objects[0].properties["content"],
        dataset_id=response.objects[0].properties["dataset_id"],
    )

collection = weaviate_client.collections.get("IRPapersText_Default")

response = hybrid_search(collection, "What is HyDE?")
print(response[0].dataset_id)

gold_doc = fetch_gold_doc(collection, response[0].dataset_id)
print(gold_doc.dataset_id)

4_5
4_5


In [49]:
from datasets import load_dataset

irpapers_queries = load_dataset("weaviate/irpapers-queries")["train"]

In [50]:
irpapers_queries[0]

{'dataset_id': '1_1',
 'pdf_id': 1,
 'pdf_title': 'Can Generative LLMs Create Query Variants for Test Collections?',
 'page_number': 1,
 'question': 'What percentage overlap in document pooling at depth 100 did GPT-3.5 generated query variants achieve compared to human-generated variants in the UQV100 test collection?',
 'answer': 'GPT-3.5 generated query variants achieved up to 71.1% overlap with human-generated variants at a pool depth of 100.'}

In [14]:
import dspy

lm = dspy.LM(
    "openai/gpt-4.1",
    cache=False,
    api_key=os.environ["OPENAI_API_KEY"]
)

dspy.configure(lm=lm, track_usage=True)

lm("say hello")

['Hello! How can I help you today? 😊']

In [42]:
from typing import Literal

# Or this could assess the relevance of an entire list of search results in one input.
# The context of the other results may ground the ratings...

'''
class AssessRelevance(dspy.Signature):
    """
    You are an impartial search relevance judge.
    Assess whether a retrieved document is relevant to a query.
    
    You are given a query, a known relevant document (labeled_gold_doc), and a 
    retrieved document to evaluate. Determine if the retrieved document would 
    satisfy the same information need as the query.
    
    Assign each result an integer relevance score from 0 to 3:
    - 3 = highly relevant, directly answers the query
    - 2 = relevant, but not perfect
    - 1 = marginally related
    - 0 = irrelevant or misleading
    """

    query: str = dspy.InputField(desc="The search query")
    labeled_gold_doc: str = dspy.InputField(desc="A document known to be relevant to the query, for reference")
    retrieved_doc: str = dspy.InputField(desc="The retrieved document to assess for relevance")
    
    relevance_score: int = dspy.OutputField(desc="The relevance score for the retrieved document")
'''

class RelevanceRanking(dspy.Signature):
    """
    You are an impartial search relevance judge.
    Given a query, a gold (human-labeled most relevant) document, and 
    a potentially noisy search system's top-ranked document, assess which document is more relevant to the query.

    If the search system's returned document is more relevant than the gold document,
    produce an improved question that better targets the gold document (so that it would be selected over the returned doc).

    IMPORTANT!! Please note the style of the original question if writing a new and improved question.
    This question should not assume that you already have access to the document and therefore should not use language such as "the document"...
    These questions are intended to test a search system's ability to retrieve relevant documents from a collection of information retrieval papers.

    Output:
    - which document is more relevant: "gold_doc" or "returned_doc"
    - improved_question (if "returned_doc" is chosen; otherwise, leave improved_question blank)
    """

    query: str = dspy.InputField(desc="The search query")
    gold_doc: str = dspy.InputField(desc="The document labeled as most relevant (gold)")
    returned_doc: str = dspy.InputField(desc="The document that the search system returned as top result")

    relevance_score: Literal["gold_doc", "returned_doc"] = dspy.OutputField(
        desc="Which document is more relevant to the query: 'gold_doc' or 'returned_doc'?"
    )
    improved_question: str = dspy.OutputField(
        desc="If 'returned_doc' is more relevant than 'gold_doc', provide a new question better answered by 'gold_doc'. Otherwise, leave blank."
    )

# update this to also output an improved question that better targets the gold doc

#relevance_assessor = dspy.ChainOfThought(AssessRelevance)

relevance_assessor = dspy.ChainOfThought(RelevanceRanking)

In [43]:
results = hybrid_search(collection, irpapers_queries[0]["question"])
print(f"Retrieved doc: {results[0].dataset_id}")

gold_doc = fetch_gold_doc(collection, irpapers_queries[0]["dataset_id"])
print(f"Gold doc: {gold_doc.dataset_id}")

response =relevance_assessor(
    query=irpapers_queries[0]["question"],
    gold_doc=gold_doc.doc,
    returned_doc=results[0].doc
)

print(response.reasoning)
print(response.relevance_score)
print(response.improved_question)

Retrieved doc: 1_4
Gold doc: 1_1
The query specifically asks for the percentage overlap in document pooling at depth 100 achieved by GPT-3.5 generated query variants compared to human-generated variants in the UQV100 test collection. The gold_doc provides the context, motivation, and general findings of the study, stating that LLM-generated query variants generate similar sets of relevant documents and mentioning "up to 71.1% overlap at a pool depth of 100," but does not provide the exact figure or the context for how it was measured.

The returned_doc directly reports the measurement: "With a temperature of 1.0, for example, the pools overlap at 43.7% at depth 10. This increases to 71.1% when examining the pools at depth 100." This explicitly answers the query by giving the exact percentage value at the correct pool depth and ties it to the experiment results. Therefore, the returned_doc is more directly relevant for providing a precise answer.
returned_doc
What is the general utility

In [44]:
irpapers_queries[0].keys()

dict_keys(['dataset_id', 'pdf_id', 'pdf_title', 'page_number', 'question', 'answer'])

In [ ]:
import time
updated_count = 0

start = time.time()
# This will take about 14 minutes for the entire dataset
for idx, query in enumerate(irpapers_queries):
    if idx % 10 == 9:
        print(f"Queries processed: {idx}.")
        print(f"Time taken so far: {time.time() - start}.")
        print(f"Queries updated: {updated_count}.")

    hybrid_search_results = hybrid_search(collection, query["question"], num_results=1)
    try:
        gold_doc = fetch_gold_doc(collection, query["dataset_id"])
        response = relevance_assessor(
            query=query["question"],
            gold_doc=gold_doc.doc,
            returned_doc=hybrid_search_results[0].doc,
        )
        if response.relevance_score == "returned_doc":
            improved_question = getattr(response, "improved_question", "")
            # Only update if improved_question is non-blank
            if improved_question:
                query["question"] = improved_question
                updated_count += 1
    except Exception as e:
        print("WARNING!!!!")
        print(f"Error fetching gold doc for query {query['dataset_id']}.")
        print(e)

print(f"Number of queries updated with improved_question: {updated_count}")
print(f"Time taken: {time.time() - start}.")

Queries processed: 9.
Time taken so far: 49.011980056762695.
Queries updated: 2.
Queries processed: 19.
Time taken so far: 107.26203894615173.
Queries updated: 3.
Queries processed: 29.
Time taken so far: 157.27206897735596.
Queries updated: 5.
Queries processed: 39.
Time taken so far: 213.54676413536072.
Queries updated: 11.
Queries processed: 49.
Time taken so far: 270.98882699012756.
Queries updated: 12.
Queries processed: 59.
Time taken so far: 333.24671483039856.
Queries updated: 12.
Queries processed: 69.
Time taken so far: 384.4067358970642.
Queries updated: 14.
Queries processed: 79.
Time taken so far: 435.5159659385681.
Queries updated: 16.
Queries processed: 89.
Time taken so far: 496.968425989151.
Queries updated: 18.
WARNING!!!!
Error fetching gold doc for query 12_9.
Queries processed: 99.
Time taken so far: 539.0167927742004.
Queries updated: 18.
Queries processed: 109.
Time taken so far: 593.1201500892639.
Queries updated: 20.
Queries processed: 119.
Time taken so far: 6

In [52]:
print("Finished!")

Finished!


# Run again with the updated dataset.

In [ ]:
import time
updated_count = 0

start = time.time()
# This will take about 14 minutes for the entire dataset
for idx, query in enumerate(irpapers_queries):
    if idx % 10 == 9:
        print(f"Queries processed: {idx}.")
        print(f"Time taken so far: {time.time() - start}.")
        print(f"Queries updated: {updated_count}.")

    hybrid_search_results = hybrid_search(collection, query["question"], num_results=1)
    try:
        gold_doc = fetch_gold_doc(collection, query["dataset_id"])
        response = relevance_assessor(
            query=query["question"],
            gold_doc=gold_doc.doc,
            returned_doc=hybrid_search_results[0].doc,
        )
        if response.relevance_score == "returned_doc":
            improved_question = getattr(response, "improved_question", "")
            # Only update if improved_question is non-blank
            if improved_question:
                query["question"] = improved_question
                updated_count += 1
    except Exception as e:
        print("WARNING!!!!")
        print(f"Error fetching gold doc for query {query['dataset_id']}.")
        print(e)

print(f"Number of queries updated with improved_question: {updated_count}")
print(f"Time taken: {time.time() - start}.")

Queries processed: 9.
Time taken so far: 47.93350887298584.
Queries updated: 2.
Queries processed: 19.
Time taken so far: 105.02587604522705.
Queries updated: 3.
Queries processed: 29.
Time taken so far: 147.68763875961304.
Queries updated: 5.
Queries processed: 39.
Time taken so far: 218.61347794532776.
Queries updated: 10.
Queries processed: 49.
Time taken so far: 267.5393478870392.
Queries updated: 11.
Queries processed: 59.
Time taken so far: 334.1000530719757.
Queries updated: 11.
Queries processed: 69.
Time taken so far: 403.3822810649872.
Queries updated: 13.
Queries processed: 79.
Time taken so far: 451.0070049762726.
Queries updated: 16.
Queries processed: 89.
Time taken so far: 505.1103138923645.
Queries updated: 18.
WARNING!!!!
Error fetching gold doc for query 12_9.
Queries processed: 99.
Time taken so far: 565.189957857132.
Queries updated: 19.
Queries processed: 109.
Time taken so far: 617.0824239253998.
Queries updated: 21.
Queries processed: 119.
Time taken so far: 688.